# Dataset creation pipeline

In this notebook, we will look at how to create a dataset with customizable variation, and how to make it into a pipeline that can be used to create datasets for e.g. machine learning purposes.


## Initial visualization
First we will explore some visualization, such that we can ensure our generated data will look how we want it to prior to creating our dataset.

The goal for this dataset is a noisy collection of disk-like structures.

In [1]:
import qim3d

# Generate initial volume collection
vol_col, labels = qim3d.generate.volume_collection(
    n_volumes = 15,
    collection_shape = (200, 200, 200),
    shape_range = ((5, 20, 20), (10, 80, 80)),
    noise_range = (0.00, 0.00),
    rotation_degree_range = (0, 360),
    rotation_axes = None,
    gamma_range = (0.9, 1),
    value_range = (128, 255),
    threshold_range = (0.5, 0.55),
    decay_rate_range = (5, 10),
    shape = 'cylinder',
    axis = 0,
    seed=0,
)

Objects placed:   0%|          | 0/15 [00:00<?, ?it/s]

We have now generated a collection of 15 volumes with varying sizes with a smaller size in the first dimension. The noise has been reduced to 0, such that they all have a disk-like structure without holes. Lastly we have changed the shape to be cylindrical. The rest of the paramters are the same as the default values.

We can now visualize it:

In [2]:
qim3d.viz.volumetric(vol_col)

Output()

And the labels:

In [3]:
qim3d.viz.volumetric(labels, colormap="hsv", opacity_function="constant")

Output()

Since we want the collection to be noisy, we can add noise with qim3d and visualize the result:

In [4]:
noisy_vol_col = qim3d.generate.background(background_shape = vol_col.shape,
                                          apply_method='add',
                                          apply_to = vol_col,
                                          seed=1)

qim3d.viz.volumetric(noisy_vol_col)

Output()

## Creating a data generation pipeline
Now that we have a baseline for how we want the data to be, we can add randomness and generate it in a loop. We can introduce more randomness by randomly selecting the amount of generated objects for each collection.

In [5]:
import random
import os

# Define the amount of datasets
num_datasets = 5

# Define and create data and label folders
base_folder = 'synthetic_dataset'
data_folder = 'data'
label_folder = 'labels'

# Make directories if they don't already exist
os.makedirs(os.path.join(base_folder, data_folder), exist_ok=True)
os.makedirs(os.path.join(base_folder, label_folder), exist_ok=True)


# Create a for-loop over the amount of datasets
for i in range(0,num_datasets):

    # Randomly select the amount of volumes in the collection
    num_volumes = random.randint(5,30)
    
    # Generate a volume with labels
    vol_col, labels = qim3d.generate.volume_collection(
        n_volumes = num_volumes,
        collection_shape = (200, 200, 200),
        shape_range = ((5, 20, 20), (10, 80, 80)),
        noise_range = (0.00, 0.00),
        rotation_degree_range = (0, 360),
        rotation_axes = None,
        gamma_range = (0.9, 1),
        value_range = (128, 255),
        threshold_range = (0.5, 0.55),
        decay_rate_range = (5, 10),
        shape = 'cylinder',
        axis = 0,
        seed = i, # Add a seed
    )
    
    # Add noise
    noisy_vol_col = qim3d.generate.background(background_shape = vol_col.shape,
                                          apply_method='add',
                                          apply_to = vol_col,
                                          seed = i # Add a seed
                                          )
    
    # Save to the different folders
    qim3d.io.save(f'{os.path.join(base_folder, data_folder)}/dataset_{i}.tif', noisy_vol_col)
    qim3d.io.save(f'{os.path.join(base_folder, label_folder)}/label_{i}.tif', labels)

Objects placed:   0%|          | 0/7 [00:00<?, ?it/s]

Objects placed:   0%|          | 0/22 [00:00<?, ?it/s]

Objects placed:   0%|          | 0/16 [00:00<?, ?it/s]

Objects placed:   0%|          | 0/9 [00:00<?, ?it/s]

Objects placed:   0%|          | 0/8 [00:00<?, ?it/s]

Note that this way of generating seeds is still replicable, and can be changed. This, however ensures that the seed is different for every generated volume.

Now we can test it and check if the volume collections look correct:

In [6]:
# Load a dataset
idx = 3

saved_collection = qim3d.io.load(f'{os.path.join(base_folder, data_folder)}/dataset_{idx}.tif')
saved_labels = qim3d.io.load(f'{os.path.join(base_folder, label_folder)}/label_{idx}.tif')

In [7]:
# Visualize a saved volume collection

qim3d.viz.volumetric(saved_collection)

Output()

In [8]:
# Visualize a saved volume label set
qim3d.viz.volumetric(saved_labels, colormap="hsv", opacity_function="constant")

Output()